# Loss function

In [27]:
import torch
inputs = torch.tensor([[16833, 3626, 6100], # "every effort moves",
                        [40, 1107, 588]])   # "I really like"

targets = torch.tensor([[3626, 6100, 345],      # "effort moves you",
                        [1107, 588, 11311]])    # "really like chocolate"

In [28]:
# place holder for GPT2 architecture implementation
import torch
import torch.nn as nn

model_config = {
    "vocab_size": 50257,
    "context_length": 256,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout": 0.1,
    "qkv_bias": False,
} 
class DummyLayerNorm(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.epsilon = 1e-5
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.epsilon)
        x = self.scale * normalized + self.shift
        return x
    
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config['embedding_dim'], 4 * config['embedding_dim']),
            nn.GELU(),
            nn.Linear(4 * config['embedding_dim'], config['embedding_dim'])
        )
    def forward(self, x):
        return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_lengh, dropout):
        super().__init__()
        print("initializing multi head attention with d_in:", d_in, "d_out:", d_out, "num_heads:", num_heads, "context_length:", context_lengh, "dropout:", dropout)
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_size = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.Dropout = nn.Dropout(dropout)
        self.OutProj = nn.Linear(d_out, d_out) # output projection to combine heads
        self.register_buffer('mask', torch.tril(torch.ones(context_lengh, context_lengh), diagonal=0)) # precompute mask for max context length of 1000

    def forward(self,x):
        batch_size, num_tokens, d_in = x.shape
        querys = self.W_query(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        keys = self.W_key(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        values = self.W_value(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        # print("querys shape:", querys.shape)
        # print("keys shape:", keys.shape)
        # print("values shape:", values.shape)
       
        querys = querys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)

        attn_scores = querys @ keys.transpose(-2,-1) # (batch_size, num_heads, num_tokens, head_size) @ (batch_size, num_heads, head_size, num_tokens) --> (batch_size, num_heads, num_tokens, num_tokens)
        # print("before masked: ", attn_scores)
        masked_attn_scores = attn_scores.masked_fill(self.mask[:num_tokens, :num_tokens] == 0, float('-inf')) # (batch_size, num_heads, num_tokens, num_tokens)
        # print("mask: ", self.mask)
        # print("after masked: ", masked_attn_scores)
        attn_weights = torch.softmax(masked_attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = self.Dropout(attn_weights) # (batch_size, num_heads, num_tokens, num_tokens)
        context_vector = attn_weights @ values # (batch_size, num_heads, num_tokens, num_tokens) @ (batch_size, num_heads, num_tokens, head_size) --> (batch_size, num_heads, num_tokens, head_size)

        context_vector = context_vector.transpose(1,2) # (batch_size, num_tokens, num_heads, head_size)
        # print("context: ", context_vector)
        #combine heads
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.num_heads * self.head_size) # (batch_size, num_tokens, d_out)
        # print("combined context: ", context_vector)

        context_vector = self.OutProj(context_vector) # (batch_size, num_tokens, d_out)
        return context_vector

    
class DummyTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(d_in=config['embedding_dim'],
            d_out=config['embedding_dim'],
            context_lengh=config['context_length'],
            num_heads=config['num_heads'],
            dropout=config['dropout'])
        self.feed_forward = FeedForward(config)
        self.norm1 = DummyLayerNorm(config['embedding_dim'])
        self.norm2 = DummyLayerNorm(config['embedding_dim'])
        self.dropout = nn.Dropout(config['dropout'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut
        return x

class DummyGPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config['vocab_size'], config['embedding_dim'])
        self.position_embedding = nn.Embedding(config['context_length'], config['embedding_dim'])
        self.dropout_embedding = nn.Dropout(config['dropout'])
        self.transformer_blocks = nn.Sequential(*[DummyTransformerBlock(config) for _ in range(config['num_layers'])])
        self.final_norm = DummyLayerNorm(config['embedding_dim'])
        self.output_head = nn.Linear(config['embedding_dim'], config['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_length = in_idx.shape
        token_embeddings = self.token_embedding(in_idx)
        position_embeddings = self.position_embedding(torch.arange(seq_length))
        x = self.dropout_embedding(token_embeddings + position_embeddings)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits



In [29]:
model = DummyGPT2(model_config)
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print(token_ids)


initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention wit

In [30]:
text_idx = 0
target_probas_1 = probas[text_idx, [0,1,2],targets[text_idx]]
print("TExt1: ", target_probas_1)

text_idx=1
target_probas_2 = probas[text_idx, [0,1,2],targets[text_idx]]
print("TExt2: ", target_probas_2)

log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

avg_log_probas = torch.mean(log_probas)
print("average log_probas: ", avg_log_probas)

neg_avg_log_probas = avg_log_probas * -1
print("neg_avg_log_probas: ", neg_avg_log_probas)

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
loss = nn.functional.cross_entropy(logits_flat, targets_flat)
print("loss: ", loss)

TExt1:  tensor([3.9400e-05, 1.5160e-05, 1.6682e-05])
TExt2:  tensor([1.3795e-05, 2.8251e-05, 3.8955e-05])
tensor([-10.1417, -11.0969, -11.0012, -11.1912, -10.4744, -10.1531])
average log_probas:  tensor(-10.6764)
neg_avg_log_probas:  tensor(10.6764)
loss:  tensor(10.6764)


In [31]:
print("open file")
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    input = f.read()
print(len(input))

open file
20479


In [32]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
total_tokens = tokenizer.encode(input)
print("Total_token: ", len(total_tokens))

Total_token:  5145


In [33]:
train_ratio = 0.9
split_idx = int(len(input)*train_ratio)
train_data = input[:split_idx]
test_data = input[split_idx:]


In [34]:

import torch
from torch.utils.data import Dataset, DataLoader

class Dataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids=[]
        self.target_ids=[]
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i+1: i+max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    token = tokenizer
    dataset = Dataset(txt, token, max_length, stride)
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,num_workers=num_workers)

    return dataloader

train_dataloader = create_dataloader(train_data, batch_size=2, max_length=model_config['context_length'],
                                     stride=model_config['context_length'], drop_last=True, shuffle=True,num_workers=0)

test_dataloader = create_dataloader(test_data, batch_size=2, max_length=model_config['context_length'],
                                     stride=model_config['context_length'], drop_last=False, shuffle=False,num_workers=0)


In [35]:
print("Trainloader")
for x,y in train_dataloader:
    print(x.shape, y.shape)

print("Testloader")
for x,y in test_dataloader:
    print(x.shape, y.shape)

Trainloader
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
Testloader
torch.Size([2, 256]) torch.Size([2, 256])


In [36]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches = None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    
    if num_batches == None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input,target) in enumerate(data_loader):
        if (i < num_batches):
            loss = calc_loss_batch(input, target, model, device)
            total_loss += loss
        else:
            break

    return total_loss/num_batches
    

In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: ", device)
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_dataloader, model, device)
    test_loss = calc_loss_loader(test_dataloader, model, device)

print("Train loss: ", train_loss)
print("Test loss: ", test_loss)

Device:  cpu
Train loss:  tensor(10.9743)
Test loss:  tensor(10.9877)


In [41]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)

    model.train()
    return train_loss, val_loss



def train_model(model, train_loader, val_loader, optimizer, device, num_epochs,
                eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_token_seen = [],[],[]
    token_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            
            token_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_token_seen.append(token_seen)
                print(f"Ep {epoch + 1} (Step: {global_step}): ",
                      f"Train loss {train_loss} ",
                      f"val_loss: {val_loss}")
                
            # generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_token_seen


In [42]:
torch.manual_seed(123)
model = DummyGPT2(model_config)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model(model,
            train_dataloader, test_dataloader, optimizer, 
            device, num_epochs, eval_freq=5, eval_iter=5, 
            start_context="Every effort moves you", tokenizer=tokenizer)


initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 256 dropout: 0.1
initializing multi head attention wit